# Day 1.1 — Your First Model Call

We begin with the smallest useful AI application:

```text
Question → Model → Response
```

This is **not yet an agent**. No tool or action loop exists.

By the end, you can call the classroom model, identify the application's role, and understand the optional local/direct-provider alternatives.

## Before you begin

### Learning outcomes

Send one prompt through the classroom route and identify request, response, provider, model, and usage fields.

Architecture reference: [D01](../../diagrams/source/day_01.md).

### Expected observation

Mock output is deterministic; live wording varies, but a non-empty response and usage record should appear.


## Concept briefing

## Why this day exists

A language model is a generator, not an application. It receives a finite context and
predicts a continuation. It does not automatically know your files, execute Python, or
continue working until a goal is complete. An agentic application is created when host
code gives the model a limited set of possible actions, carries state between turns,
executes approved actions, and decides when the run must stop.

Day 1 removes the apparent magic from this process. By the end, students should be able
to point to the exact line that sends a request, the exact data that describes a tool,
the exact function that executes it, and the exact condition that terminates the loop.

## What a model call actually contains

A typical request contains a model identifier, ordered messages, optional tool schemas,
and generation controls. Messages are not merely a chat transcript. Their roles tell the
provider how each piece should be interpreted:

- `system`: standing instructions and boundaries;
- `user`: the current task or supplied information;
- `assistant`: previous model output, including tool requests;
- `tool`: an observation produced by host-executed code.

The provider serializes this request into a form the model can process. The model sees
tokens representing instructions, messages and tool descriptions. It does not receive a
live Python function. When it appears to "call" a tool, it is generating structured
tokens that name a function and propose arguments. The host application parses those
tokens, validates the arguments, applies policy, calls ordinary code, and returns the
result in another message.

This distinction is load-bearing:

```text
model proposes structured tokens
-> application validates and authorizes
-> Python executes
-> application records the observation
-> model sees the observation on the next call
```

If the model invents a tool name, supplies the wrong type, or requests a prohibited
action, nothing should happen unless the application accepts the request.


## The four course routes

1. **OpenRouter + GPT-OSS:** primary classroom route using your individually issued key.
2. **Ollama:** optional local-provider comparison for suitable computers.
3. **Direct OpenAI API:** optional for students with their own API access.
4. **Mock mode:** deterministic testing without network calls or cost.

The remaining guided notebooks use OpenRouter consistently. Never paste a key into a notebook or commit `.env`.

## Part A — OpenRouter (classroom default)

Before class, place your issued key in the repository's `.env` file:

```dotenv
OPENROUTER_API_KEY=your_individual_course_key
OPENROUTER_MODEL=openai/gpt-oss-120b
```

Your key has a course-wide lifetime spending limit. Do not share it.

In [ ]:
# Run once if required, then restart the kernel.
# %pip install -q openai python-dotenv
import os
from types import SimpleNamespace
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
COURSE_MODEL=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key) if api_key else None
print("Route:","OpenRouter" if client else "mock fallback")


In [ ]:
question="Explain recursion in two sentences for a beginner."
if client:
    response=client.chat.completions.create(model=COURSE_MODEL,messages=[{"role":"user","content":question}],
        max_tokens=300,extra_body={"reasoning":{"effort":"low","exclude":True},"provider":{"sort":"price"}})
    answer=response.choices[0].message.content
else:
    response=None
    answer="Recursion is when a function solves a problem by calling itself on a smaller version. It needs a base case so the calls eventually stop."
print(answer)


### Observe

- Which object sends the request?
- Which value selects the model?
- Where is response length bounded?
- Did the model execute a Python function?
- Why is low reasoning sufficient for this simple request?

The application sends messages and controls the request. The model generates text.

In [ ]:
if response:
    print(response.usage)
else:
    print({"provider":"mock","prompt_tokens":0,"completion_tokens":0,"cost_usd":0.0})


## Part B — Ollama (optional provider-portability comparison)

If your computer can run a local model, install Ollama and its Python package, download the instructor-approved comparison model, and run the same prompt. This section is optional; the course does not assume every laptop can run it well.

In [ ]:
# Optional local route:
# %pip install -q ollama
# from ollama import chat
# local_response = chat(
#     model="qwen3:4b",  # replace with the instructor-approved comparison model
#     messages=[{"role": "user", "content": question}],
# )
# print(local_response.message.content)

## Part C — Direct OpenAI API (optional alternative)

Students with their own OpenAI API project can use the official Python SDK and Responses API. A ChatGPT subscription and API billing are separate. The SDK reads `OPENAI_API_KEY`; never write the key in this notebook.

Official guide: https://platform.openai.com/docs/quickstart

In [ ]:
# Optional direct OpenAI route:
# from openai import OpenAI
# direct_client = OpenAI()  # reads OPENAI_API_KEY
# direct_response = direct_client.responses.create(
#     model=os.getenv("OPENAI_MODEL", "gpt-5.6-luna"),
#     input=question,
# )
# print(direct_response.output_text)

## Part D — Mock mode

A mock is useful for testing Python flow during an outage, but it does not measure real model quality.

In [ ]:
def mock_model(prompt: str) -> str:
    return f"Prepared mock response for: {prompt}"

print(mock_model(question))

## Exercise and checkpoint

Ask for a two-sentence explanation, one example, and one limitation from your engineering discipline. Run it twice and note what changes.

We built `application → model → text response`. The limitation is that free-form text is not a dependable application data structure.

## Required live observation

Send one bounded prompt through the issued OpenRouter route and save the response plus usage record. If service access fails, inspect the instructor-captured trace and continue in mock mode.


## Your turn

Change one prompt constraint and compare outputs without changing providers.

## Recap

A model call generates output from supplied context; it does not create an agent.
